# Prompt 31: Practice Exam Questions
## Databricks Certified Associate Developer for Apache Spark
### 20 Multiple-Choice Questions — Full Exam Coverage

---

**Instructions:** Each question has four options (A–D). Select the best answer. Reveal the answer and explanation by expanding the `Answer & Explanation` section below each question.

**Distribution:**
- Questions 1–4: Spark Architecture & Internals
- Questions 5–8: Spark SQL & Built-in Functions
- Questions 9–14: DataFrame API
- Questions 15–16: Troubleshooting & Performance Tuning
- Questions 17–18: Structured Streaming
- Question 19: Spark Connect
- Question 20: Pandas API on Apache Spark

---

## Section 1 — Spark Architecture & Internals (Questions 1–4)

### Q1 (Architecture — Conceptual)

A Spark application has a Driver process and multiple Executor processes. Which of the following statements correctly describes the role of the Driver?

**A.** The Driver executes user-defined functions and applies transformations to individual data partitions.

**B.** The Driver orchestrates the job by creating a DAG, negotiating resources with the Cluster Manager, and scheduling Tasks on Executors.

**C.** The Driver stores the shuffle intermediate data between stages and coordinates data movement across Executors.

**D.** The Driver maintains a persistent connection to all Executors and forwards task results to the Cluster Manager.

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

The Driver's responsibilities are:
- Convert the user's code into a **DAG** (Directed Acyclic Graph) of Stages and Tasks
- Negotiate resources with the **Cluster Manager** (YARN, Mesos, Kubernetes, or Standalone)
- **Schedule Tasks** on Executors via the TaskScheduler
- Collect and aggregate results from Executors
- Maintain the `SparkContext`

**Why the others are wrong:**
- **A** is wrong — executing transformations on data partitions is the **Executor's** job, not the Driver's
- **C** is wrong — shuffle data is stored on the Executor's local disk, not the Driver
- **D** is wrong — the Driver communicates with Executors but does not "forward results to the Cluster Manager"
</details>

---

### Q2 (Architecture — Lazy Evaluation)

Given the following code, how many times does Spark actually read the CSV file?

```python
df = spark.read.csv('/data/transactions.csv', header=True)
df_filtered = df.filter(df['amount'] > 100)
df_grouped = df_filtered.groupBy('category').count()
df_grouped.show()
df_grouped.show()
```

**A.** 0 — Spark does not read files directly; it uses a metadata catalog.

**B.** 1 — The file is read once when `spark.read.csv()` is called.

**C.** 2 — The file is read once per `show()` action because `df_grouped` is not cached.

**D.** 4 — The file is read once per transformation and once per action.

<details>
<summary>Answer & Explanation</summary>

**Answer: C**

Due to **lazy evaluation**, `spark.read.csv()`, `.filter()`, and `.groupBy().count()` are all **transformations** — they build a logical plan but do not execute.

Execution is triggered by **actions**. `show()` is an action. Since `df_grouped` is **not cached**, calling `show()` twice causes Spark to **re-execute the full lineage twice**, reading the CSV file both times.

**Fix — cache to avoid re-reading:**
```python
df_grouped.cache()    # or .persist()
df_grouped.show()     # first show: reads CSV, computes, caches result
df_grouped.show()     # second show: reads from cache — no CSV re-read
```

**Why the others are wrong:**
- **A** is wrong — Spark does read files
- **B** is wrong — `.read.csv()` is lazy; no read happens at that line
- **D** is wrong — transformations are lazy and do not trigger reads
</details>

---

### Q3 (Architecture — Shuffling)

A developer notices that a Spark job with a `groupBy` on a high-cardinality column is extremely slow. Which of the following is the most likely cause, and what is the recommended fix?

**A.** The job is slow because `groupBy` uses lazy evaluation. Fix: call `.cache()` before `groupBy`.

**B.** The job is slow because `groupBy` triggers a shuffle, causing all records with the same key to be moved across the network. Fix: increase `spark.sql.shuffle.partitions` to improve parallelism.

**C.** The job is slow because the default shuffle partition count (200) creates too many tasks. Fix: reduce `spark.sql.shuffle.partitions` to match the data size and cluster capacity.

**D.** The job is slow because `groupBy` requires a broadcast join. Fix: add `broadcast()` hint to the DataFrame.

<details>
<summary>Answer & Explanation</summary>

**Answer: C**

The default value of `spark.sql.shuffle.partitions` is **200**. For many workloads, especially on smaller clusters or smaller datasets, 200 shuffle partitions results in:
- Many tiny tasks with high scheduling overhead
- Potential for thousands of small files in output
- Memory pressure from too many concurrent tasks

The recommended fix is to tune this value based on data size and cluster capacity:
```python
spark.conf.set('spark.sql.shuffle.partitions', '20')  # for smaller datasets
# Rule of thumb: aim for 128–256 MB per partition after the shuffle
```

**Note:** B is partially correct (groupBy does trigger a shuffle) but the suggested fix (increasing the partition count) would make the problem worse, not better. C correctly identifies both the cause and the fix.

**Why the others are wrong:**
- **A** is wrong — caching before groupBy doesn't fix shuffle overhead
- **D** is wrong — `groupBy` is not a join; broadcast hints apply to joins
</details>

---

### Q4 (Architecture — Fault Tolerance)

A Spark job is running on a 10-node cluster. Partway through execution, one Executor node fails. Which statement best describes what happens?

**A.** The entire job fails because Spark cannot recover from Executor failures.

**B.** The job pauses until the failed node is restarted and rejoins the cluster.

**C.** Spark automatically re-schedules the failed tasks on surviving Executors using RDD lineage (or the DataFrame plan) to recompute the lost partitions.

**D.** Spark checkpoints the entire job state every 5 minutes, so only the last 5 minutes of work is lost.

<details>
<summary>Answer & Explanation</summary>

**Answer: C**

Spark's fault tolerance is based on **RDD lineage** (the dependency graph of transformations). When an Executor fails:
1. The Driver detects the failure via the heartbeat mechanism
2. The Driver identifies which **partitions** were being processed by the failed Executor
3. Spark **recomputes** those specific partitions on surviving Executors by re-executing the lineage from the last available data source or checkpoint
4. The job continues — no full restart needed

This recomputation only applies to the lost partitions, not the entire dataset.

**Why the others are wrong:**
- **A** is wrong — Spark recovers from Executor failures automatically
- **B** is wrong — Spark does not wait for the failed node; it reassigns work
- **D** is wrong — automatic checkpointing every 5 minutes is not a Spark feature (checkpointing is manual and configurable, mainly for streaming)
</details>

---

## Section 2 — Spark SQL & Built-in Functions (Questions 5–8)

### Q5 (Spark SQL — Window Functions)

A developer wants to rank employees within each department by salary (highest salary = rank 1), with no gaps in rank values even when multiple employees have the same salary. Which window function should be used?

**A.** `rank()` — assigns the same rank to ties, with gaps after tied ranks (1, 1, 3, 4...)

**B.** `dense_rank()` — assigns the same rank to ties, with NO gaps (1, 1, 2, 3...)

**C.** `row_number()` — assigns a unique sequential number to every row regardless of ties

**D.** `ntile(4)` — divides rows into 4 equal groups

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

The requirement specifies "no gaps in rank values" — this is exactly what `dense_rank()` provides.

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window = Window.partitionBy('department').orderBy(desc('salary'))
df.withColumn('rank', dense_rank().over(window)).show()
```

**Comparison:**

| Salary | `rank()` | `dense_rank()` | `row_number()` |
|--------|---------|---------------|---------------|
| 100000 | 1       | 1             | 1             |
| 100000 | 1       | 1             | 2             |
| 90000  | 3       | **2**         | 3             |
| 80000  | 4       | **3**         | 4             |

`rank()` skips to 3 after two employees tied at rank 1. `dense_rank()` continues with 2. The question specifically says "no gaps" → `dense_rank()`.

**Why the others are wrong:**
- **A** is wrong — `rank()` has gaps after ties
- **C** is wrong — `row_number()` gives unique values, so tied employees get different ranks
- **D** is wrong — `ntile()` divides rows into buckets, not a ranking function
</details>

---

### Q6 (Spark SQL — Query Optimisation)

A developer writes the following SQL query. What optimisation does the Spark Catalyst optimizer most likely apply automatically?

```sql
SELECT e.name, e.salary, d.department_name
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
WHERE e.salary > 80000
  AND d.country = 'USA'
```

**A.** Catalyst rewrites the join as a cross join to improve parallelism.

**B.** Catalyst pushes the `WHERE` filters down to execute before the join, reducing the number of rows that participate in the join.

**C.** Catalyst automatically converts the SQL query to RDD operations for better performance.

**D.** Catalyst caches both tables in memory before executing the join.

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

This is **predicate pushdown** — one of Catalyst's most important optimisations. Instead of:
1. Read all of `employees`
2. Read all of `departments`
3. JOIN (large result)
4. FILTER WHERE salary > 80000 AND country = 'USA'

Catalyst reorders to:
1. Read `employees` WITH filter `salary > 80000` (fewer rows)
2. Read `departments` WITH filter `country = 'USA'` (fewer rows)
3. JOIN the already-filtered smaller datasets

When reading from Parquet, Delta Lake, or other columnar/partitioned formats, predicate pushdown can further reduce I/O by only reading the necessary row groups or partitions.

**Why the others are wrong:**
- **A** is wrong — converting to a cross join would be catastrophically wrong
- **C** is wrong — Catalyst works at the DataFrame/SQL level; it does not convert to RDDs
- **D** is wrong — automatic caching is not a Catalyst optimisation (caching is a manual action)
</details>

---

### Q7 (Spark SQL — Built-in Functions)

A developer needs to extract the year and month from a timestamp column `ts` of type `TimestampType`. Which expression correctly extracts both?

**A.**
```python
df.withColumn('yr', col('ts').year).withColumn('mo', col('ts').month)
```

**B.**
```python
from pyspark.sql.functions import year, month
df.withColumn('yr', year(col('ts'))).withColumn('mo', month(col('ts')))
```

**C.**
```python
df.withColumn('yr', ts.getYear()).withColumn('mo', ts.getMonth())
```

**D.**
```python
df.select('ts').map(lambda r: r.ts.year)
```

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

Spark provides `year()` and `month()` as built-in functions in `pyspark.sql.functions`. They accept a Column of `DateType` or `TimestampType` and return an `IntegerType` column.

```python
from pyspark.sql.functions import year, month, dayofmonth, hour, minute, second

df.withColumn('yr',  year(col('ts'))) \
  .withColumn('mo',  month(col('ts'))) \
  .withColumn('day', dayofmonth(col('ts'))) \
  .withColumn('hr',  hour(col('ts')))
```

**Why the others are wrong:**
- **A** is wrong — Spark Column objects do not have `.year` or `.month` attributes (Python datetime objects do, but not Spark Column objects)
- **C** is wrong — `.getYear()` is not a valid Spark Column method
- **D** is wrong — `.map()` is an RDD operation; this converts to RDD and uses Python datetime attributes, which works but is inefficient and loses the DataFrame API
</details>

---

### Q8 (Spark SQL — NULL Handling)

A DataFrame has a column `score` that contains both `NULL` values and Python `float('nan')` (NaN) values. A developer runs:

```python
df.na.fill({'score': 0})
```

What is the result?

**A.** Both NULL and NaN values in `score` are replaced with 0.

**B.** Only NULL values are replaced with 0; NaN values remain as NaN.

**C.** Only NaN values are replaced with 0; NULL values remain as NULL.

**D.** An error is raised because `fill()` does not support numeric columns.

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

`df.na.fill()` (and `df.na.drop()`) operate only on **SQL NULL** values. They do **not** recognise or process **NaN** (`float('nan')`) values because NaN is a special floating-point value, not a SQL NULL.

To handle NaN values, you must first convert NaN to NULL:
```python
from pyspark.sql.functions import isnan, when, col

# Step 1: Convert NaN to NULL
df_clean = df.withColumn('score',
    when(isnan(col('score')), None).otherwise(col('score'))
)

# Step 2: Now fill() handles both (originally-NULL and converted-from-NaN)
df_clean.na.fill({'score': 0})
```

Or in one step:
```python
from pyspark.sql.functions import nanvl
df.withColumn('score', nanvl(col('score'), lit(0)))  # replaces NaN with 0, leaves NULL
.na.fill({'score': 0})  # then handle NULL
```

**Key rule:** `dropna()` / `fillna()` / `na.fill()` / `na.drop()` → handle **NULL** only. `isnan()` / `nanvl()` → handle **NaN** only.
</details>

---

## Section 3 — DataFrame API (Questions 9–14)

### Q9 (DataFrame API — Code Reading)

What does the following code output?

```python
from pyspark.sql.functions import col, when

data = [(1, 'Alice', 85), (2, 'Bob', None), (3, 'Carol', 72), (4, 'Dave', None)]
df = spark.createDataFrame(data, ['id', 'name', 'score'])

df2 = df.withColumn('grade',
    when(col('score') >= 90, 'A')
    .when(col('score') >= 75, 'B')
    .when(col('score') >= 60, 'C')
    .otherwise('F')
)
df2.select('name', 'score', 'grade').show()
```

**A.**
```
Alice | 85   | B
Bob   | null | F
Carol | 72   | C
Dave  | null | F
```

**B.**
```
Alice | 85   | B
Bob   | null | null
Carol | 72   | C
Dave  | null | null
```

**C.**
```
Alice | 85   | B
Bob   | null | B
Carol | 72   | C
Dave  | null | B
```

**D.** An error is raised because `col('score') >= 90` throws an exception when `score` is NULL.

<details>
<summary>Answer & Explanation</summary>

**Answer: A**

In Spark SQL (and in the `when()` function), comparisons involving `NULL` evaluate to `NULL` (unknown), not `true` or `false`. So:
- For Bob: `NULL >= 90` → NULL (not true), `NULL >= 75` → NULL (not true), `NULL >= 60` → NULL (not true)
- Since all conditions evaluate to NULL/false, the `.otherwise('F')` branch is taken
- Result: Bob's grade is **'F'**

**Why B is wrong:** `otherwise()` is provided, so NULL-comparisons fall through to `.otherwise('F')` — not `null`. B would be correct if `.otherwise()` was not specified (then unmatched rows produce NULL).

**Why D is wrong:** Spark does not raise an exception for NULL comparisons — it evaluates them as NULL (three-valued logic).

**Key rule:** In Spark, `NULL >= any_value` → NULL → treated as false in a `when()` chain → falls to `otherwise()` if provided.
</details>

---

### Q10 (DataFrame API — Joins)

A developer joins two DataFrames and gets more rows in the result than expected. The join is:

```python
result = customers.join(orders, on='customer_id', how='inner')
print(result.count())  # much larger than customers.count()
```

What is the most likely cause?

**A.** An inner join always produces more rows than the left DataFrame — this is expected behavior.

**B.** The `orders` DataFrame contains multiple rows per `customer_id`, so each customer row is duplicated for every matching order row.

**C.** Spark performs a Cartesian product for inner joins by default.

**D.** There are NULL values in `customer_id` — NULL joins with every row.

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

This is the **row multiplication (fan-out) problem** caused by a one-to-many or many-to-many join relationship.

If a customer has 5 orders, the inner join produces **5 rows** for that customer (one per order). If customers have an average of 10 orders, the result has ~10× more rows than the customers table.

**Example:**
```
customers: 100 rows (one row per customer)
orders: 1000 rows (avg 10 orders per customer)
result: 1000 rows (each customer row appears once per order)
```

**Fix — if you only want unique customers with any order:**
```python
result = customers.join(orders.select('customer_id').distinct(),
                        on='customer_id', how='inner')
```

**Why the others are wrong:**
- **A** is wrong — an inner join can produce fewer or more rows depending on the relationship
- **C** is wrong — Spark does not perform Cartesian products for inner joins (that would require `crossJoin()`)
- **D** is wrong — NULL values in join keys do NOT match each other in Spark (NULL != NULL in SQL semantics)
</details>

---

### Q11 (DataFrame API — Repartition vs Coalesce)

A developer has a DataFrame with 200 partitions after a shuffle. They want to write it as Parquet and only need 10 output files. Which operation is most appropriate?

**A.** `df.repartition(10).write.parquet('/output/')`

**B.** `df.coalesce(10).write.parquet('/output/')`

**C.** `df.repartition(10).coalesce(10).write.parquet('/output/')`

**D.** `df.write.option('numPartitions', 10).parquet('/output/')`

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

`coalesce(10)` is the correct choice when **reducing** the number of partitions because:
- It is a **narrow transformation** — it combines existing partitions without a full shuffle
- It is **much cheaper** than `repartition()` for reducing partition count
- It cannot increase the number of partitions (can only reduce)

`repartition(10)` would also produce 10 output files but:
- It triggers a **full shuffle** — all data is redistributed across the network
- This is unnecessarily expensive when going from 200 → 10 (reduction)

**Rule:**
- **Reducing** partitions → `coalesce()` (no shuffle, efficient)
- **Increasing** partitions → `repartition()` (requires shuffle)
- **Evenly distributing** data → `repartition()` (coalesce can produce skewed partition sizes)

**Why the others are wrong:**
- **A** is wrong — `repartition(10)` works but is inefficient (unnecessary full shuffle)
- **C** is wrong — doing both is redundant and wasteful
- **D** is wrong — `numPartitions` is not a valid write option for Parquet in PySpark
</details>

---

### Q12 (DataFrame API — UDFs)

A developer defines a UDF to categorise salaries:

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def tier(salary):
    if salary > 100000: return 'High'
    if salary > 60000:  return 'Mid'
    return 'Low'

tier_udf = udf(tier, StringType())
df.withColumn('tier', tier_udf('salary')).show()
```

The dataset has 10 rows where `salary` is NULL. What does the `tier` column contain for those rows?

**A.** `'Low'` — because NULL fails both `> 100000` and `> 60000` checks, so the function returns `'Low'`.

**B.** `NULL` — Spark automatically returns NULL for any UDF when the input is NULL, without calling the function.

**C.** An exception is raised because Python cannot compare `None > 100000`.

**D.** `'Low'` — Spark converts NULL to 0 before passing it to Python UDFs.

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

By default, Spark UDFs treat NULL inputs with **NULL propagation** — if the input is NULL, the UDF returns NULL without invoking the Python function. This is the default behavior for Spark UDFs.

However, this is configurable:
```python
# Default behavior: NULL input → NULL output (function NOT called)
tier_udf = udf(tier, StringType())

# To handle NULL inside the UDF:
def tier_null_safe(salary):
    if salary is None: return 'Unknown'
    if salary > 100000: return 'High'
    if salary > 60000:  return 'Mid'
    return 'Low'

tier_udf_safe = udf(tier_null_safe, StringType())
# Still: NULL input → function IS called with salary=None
```

**Important:** In Python, `None > 100000` raises a `TypeError` in Python 3. If the UDF WERE called with None, it would fail. This is why Spark's default NULL propagation (not calling the function) is important for safety.

**Why the others are wrong:**
- **A** is wrong — the function is NOT called when input is NULL
- **C** is wrong — Spark does NOT call the UDF for NULL inputs by default
- **D** is wrong — Spark does not convert NULL to 0
</details>

---

### Q13 (DataFrame API — Reading/Writing)

A developer needs to write a DataFrame to Parquet, partitioned by the `country` column, and wants to overwrite **only the partitions that are present in the new data** without affecting other existing partitions. Which write mode and option should be used?

**A.**
```python
df.write.mode('overwrite').partitionBy('country').parquet('/output/')
```

**B.**
```python
df.write.mode('append').partitionBy('country').parquet('/output/')
```

**C.**
```python
spark.conf.set('spark.sql.sources.partitionOverwriteMode', 'dynamic')
df.write.mode('overwrite').partitionBy('country').parquet('/output/')
```

**D.**
```python
df.write.mode('overwrite').option('overwriteSchema', 'true').partitionBy('country').parquet('/output/')
```

<details>
<summary>Answer & Explanation</summary>

**Answer: C**

By default, `mode('overwrite')` with a partitioned write **deletes the entire output directory** and rewrites everything — even partitions not in the new data.

To overwrite only the specific partitions present in the new DataFrame (preserving other partitions), set:
```python
spark.conf.set('spark.sql.sources.partitionOverwriteMode', 'dynamic')
```

With `partitionOverwriteMode = dynamic`:
- Only partitions that appear in the new data are overwritten
- Partitions NOT in the new data are left untouched

**Example use case:** Daily data load — overwrite today's partition without touching yesterday's or any other day.

**Why the others are wrong:**
- **A** is wrong — default `mode('overwrite')` deletes ALL existing partitions and rewrites from scratch
- **B** is wrong — `append` adds new data to existing partitions, causing duplicates
- **D** is wrong — `overwriteSchema` is a Delta Lake option for schema changes, not standard Parquet partition control
</details>

---

### Q14 (DataFrame API — Aggregations)

What does the following code do, and what is the output type of `result`?

```python
from pyspark.sql.functions import sum, avg, max, countDistinct

result = df.agg(
    sum('revenue').alias('total_revenue'),
    avg('revenue').alias('avg_revenue'),
    max('revenue').alias('max_revenue'),
    countDistinct('category').alias('num_categories')
)
```

**A.** `result` is a Python `dict` with keys `total_revenue`, `avg_revenue`, `max_revenue`, `num_categories`.

**B.** `result` is a Spark `Row` object with the four aggregated values.

**C.** `result` is a Spark `DataFrame` with a single row and four columns.

**D.** `result` is a Spark `DataFrame` with one row per distinct `category` value.

<details>
<summary>Answer & Explanation</summary>

**Answer: C**

`df.agg()` called directly on a DataFrame (without a preceding `groupBy()`) applies the aggregation functions to the **entire DataFrame** and returns a new **DataFrame** with a single row and one column per aggregation.

```python
result.show()
# +-------------+-----------+-----------+--------------+
# |total_revenue|avg_revenue|max_revenue|num_categories|
# +-------------+-----------+-----------+--------------+
# |     1234567.|    5678.9 |   99999.0 |            8 |
# +-------------+-----------+-----------+--------------+

# To get a Python value:
total = result.collect()[0]['total_revenue']  # collect → Row → dict access
# OR:
total = result.first()['total_revenue']
```

**Why the others are wrong:**
- **A** is wrong — `.agg()` returns a DataFrame, not a dict
- **B** is wrong — `.agg()` returns a DataFrame (you'd get a Row after `.first()` or `.collect()[0]`)
- **D** is wrong — without `groupBy()`, `.agg()` produces one row for the whole dataset; with `groupBy('category').agg()` you'd get one row per category
</details>

---

## Section 4 — Troubleshooting & Performance Tuning (Questions 15–16)

### Q15 (Troubleshooting — Data Skew)

A Spark job has 100 tasks in a shuffle stage. 99 tasks complete in 10 seconds each, but one task takes 15 minutes. What is the most likely cause?

**A.** The cluster does not have enough memory for the task — it is spilling to disk.

**B.** The key used for the shuffle has a highly skewed distribution — one key has far more records than others, creating a "hot partition".

**C.** The Spark driver is overloaded and cannot schedule the remaining task.

**D.** The task is blocked waiting for another stage to complete.

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

This is the classic symptom of **data skew** — one partition contains a disproportionately large amount of data because many records share the same key value. For example, if 80% of orders belong to a single customer_id, that partition will have 80× more data than average.

**Diagnosis:** In the Spark UI, the "Tasks" tab for the slow stage will show one task with significantly more input data than the others.

**Solutions:**

1. **Salting** — add a random suffix to the skewed key to distribute records
```python
from pyspark.sql.functions import concat, col, lit, floor, rand
df_salted = df.withColumn('key_salted',
    concat(col('skewed_key'), lit('_'), (rand() * 10).cast('int').cast('string'))
)
```

2. **Skew hint** (Spark 3.x AQE)
```python
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
```

3. **Broadcast join** — if one side is small enough, broadcast it to avoid shuffle entirely

**Why the others are wrong:**
- **A** is possible but would show spill metrics, not a single slow task
- **C** is wrong — driver overload affects scheduling of all tasks, not just one
- **D** is wrong — if other tasks in the same stage are completing, there is no inter-stage dependency
</details>

---

### Q16 (Performance Tuning — Broadcast Join)

A join between a 500 GB table and a 10 MB lookup table is running slowly due to shuffle. What is the most effective optimisation?

**A.** Increase `spark.sql.shuffle.partitions` to 2000.

**B.** Repartition the large table by the join key before the join.

**C.** Use a broadcast join to send the small table to all Executors, eliminating the shuffle.

**D.** Convert both DataFrames to RDDs before joining, using `rdd.join()`.

<details>
<summary>Answer & Explanation</summary>

**Answer: C**

A **broadcast join** (also called a map-side join or replicated join) sends the small table to every Executor node so that the join can be performed locally without a shuffle. This is ideal when one side of the join is small enough to fit in Executor memory.

```python
from pyspark.sql.functions import broadcast

result = large_df.join(broadcast(small_lookup_df), on='join_key', how='inner')
# No shuffle for the small table — copied to every Executor
```

**Automatic broadcast:** Spark will automatically broadcast tables below `spark.sql.autoBroadcastJoinThreshold` (default: 10 MB). Since the lookup table is exactly 10 MB, Spark may auto-broadcast — but an explicit `broadcast()` hint ensures it.

```python
# Increase threshold if needed:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '50mb')
```

**Why the others are wrong:**
- **A** is wrong — more shuffle partitions means more overhead for this join, not less
- **B** is wrong — repartitioning reduces the shuffle for subsequent operations but still requires a shuffle for the repartition itself
- **D** is wrong — converting to RDD removes Catalyst optimisations and makes things worse
</details>

---

## Section 5 — Structured Streaming (Questions 17–18)

### Q17 (Streaming — Output Modes)

A streaming query reads from a Kafka topic and computes a running total of purchases per user. The developer wants each micro-batch to write **all rows** (including previously written rows with updated totals) to a Delta Lake table. Which output mode should be used?

**A.** `append` — writes only the new rows from each micro-batch.

**B.** `complete` — writes the full result table after every micro-batch (requires aggregation).

**C.** `update` — writes only the rows that have changed since the last micro-batch.

**D.** `replace` — replaces all existing rows with the current result.

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

The three streaming output modes are:

| Mode | What is written | When to use |
|------|----------------|-------------|
| `append` | Only NEW rows that will never change | Simple row-by-row processing, no aggregations that can be updated |
| `complete` | The ENTIRE result table after each batch | Aggregations where every batch produces a full updated result |
| `update` | Only rows that CHANGED since last batch | Aggregations with state — more efficient than `complete` for large result tables |

The requirement is to write **all rows** (including updated totals) after each micro-batch → **`complete`** mode.

Note: `complete` mode requires an aggregation in the query (otherwise Spark cannot determine what the "complete" result is).

```python
query = streaming_df \
    .groupBy('user_id') \
    .agg(sum('purchase').alias('total_purchase')) \
    .writeStream \
    .outputMode('complete') \
    .format('delta') \
    .option('checkpointLocation', '/ckpt/') \
    .start('/output/user_totals/')
```

**Why the others are wrong:**
- **A** is wrong — `append` only writes new rows; with a running total, old rows are updated, not new
- **C** is wrong — `update` only writes changed rows, not all rows
- **D** is wrong — `replace` is not a valid Spark streaming output mode
</details>

---

### Q18 (Streaming — Watermarking)

A Structured Streaming query uses a 10-minute watermark on an event-time column. An event arrives that is 15 minutes late (its event-time is 15 minutes behind the current watermark). What happens to this event?

**A.** The event is processed normally — watermarks only affect output, not processing.

**B.** The event is placed in a buffer and processed in the next micro-batch.

**C.** The event is dropped — it arrived after the watermark cutoff.

**D.** The event triggers a full recomputation of all previous micro-batches.

<details>
<summary>Answer & Explanation</summary>

**Answer: C**

A watermark defines the **tolerated event-time lag**. Events that arrive later than the watermark threshold are considered **too late** and are **dropped** (not processed).

How the watermark works:
```
Watermark = max_event_time_seen - watermark_delay
```

If the watermark is set to 10 minutes and the current watermark boundary is 12:00, then:
- An event with event-time 11:55 (5 min late) → **processed** (within 10-min window)
- An event with event-time 11:40 (20 min late) → **dropped** (beyond 10-min window)

In this question, the event is 15 minutes late (beyond the 10-minute watermark) → **dropped**.

```python
df.withWatermark('event_time', '10 minutes') \
  .groupBy(
      window(col('event_time'), '5 minutes'),
      col('user_id')
  ) \
  .count()
```

**Trade-off:** Larger watermark = more late data accepted = more state held in memory. Smaller watermark = less memory use = more data dropped.

**Why the others are wrong:**
- **A** is wrong — watermarks affect BOTH state cleanup AND late data processing
- **B** is wrong — buffering late data indefinitely would prevent state cleanup
- **D** is wrong — Spark does not recompute previous micro-batches for late data (that's the point of watermarks — to define a hard cutoff)
</details>

---

## Section 6 — Spark Connect (Question 19)

### Q19 (Spark Connect — Conceptual)

A data scientist wants to connect to a remote Spark Connect server running at `spark.company.com` on the default port. They also want to confirm which operations are NOT supported via Spark Connect. Which of the following statements is correct?

**A.**
```python
spark = SparkSession.builder.master('spark://spark.company.com:15002').getOrCreate()
# SparkContext is accessible via spark.sparkContext
```

**B.**
```python
spark = SparkSession.builder.remote('sc://spark.company.com:15002').getOrCreate()
# spark.sparkContext raises an error — SparkContext not accessible client-side
```

**C.**
```python
spark = SparkSession.builder.remote('sc://spark.company.com:15002').getOrCreate()
# RDD API works fine via Spark Connect
```

**D.**
```python
spark = SparkSession.builder.remote('spark://spark.company.com:15002').getOrCreate()
# DataFrame API and SQL are not supported via Spark Connect
```

<details>
<summary>Answer & Explanation</summary>

**Answer: B**

Spark Connect connection:
- **Method:** `.remote()` (not `.master()`)
- **Protocol prefix:** `sc://` (not `spark://` which connects to a Standalone cluster manager)
- **Default port:** `15002`

NOT supported via Spark Connect (client-side):
- `spark.sparkContext` — the driver runs on the server; there is no SparkContext on the client
- RDD API — `sc.parallelize()`, `sc.textFile()`, `rdd.map()`, etc.
- Accumulators and broadcast variables via SparkContext
- `spark._jvm` (Py4J JVM access)

Fully supported:
- DataFrame API
- Spark SQL
- UDFs
- Reading/writing data sources
- Streaming

**Why the others are wrong:**
- **A** uses `.master()` not `.remote()`, and incorrectly states SparkContext is accessible
- **C** uses correct connection syntax but incorrectly states RDD API works
- **D** uses wrong protocol (`spark://` not `sc://`) and incorrectly states DataFrame/SQL are unsupported
</details>

---

## Section 7 — Pandas API on Apache Spark (Question 20)

### Q20 (Pandas API on Spark — Code Reading)

A developer writes the following code:

```python
import pyspark.pandas as ps

sdf = spark.read.parquet('/data/large_table/')  # 50 GB of data
psdf = sdf.pandas_api()
pdf = psdf.groupby('region').agg({'revenue': 'sum'}).to_pandas()
print(pdf)
```

Which statement best describes the performance characteristics of this code?

**A.** The entire 50 GB dataset is immediately collected to the driver when `sdf.pandas_api()` is called.

**B.** `psdf.groupby('region').agg({'revenue': 'sum'})` is executed in-memory on the driver machine. Only `to_pandas()` triggers distributed processing.

**C.** `sdf.pandas_api()` and the `groupby().agg()` are lazy operations on the Spark cluster. `to_pandas()` triggers execution and collects only the aggregated result (likely a small number of rows) to the driver.

**D.** The `import pyspark.pandas as ps` statement starts a new SparkSession. The existing `spark` session is not used.

<details>
<summary>Answer & Explanation</summary>

**Answer: C**

Step-by-step analysis:

1. `sdf = spark.read.parquet(...)` — lazy; builds a plan to read 50 GB, no data movement
2. `psdf = sdf.pandas_api()` — **no data movement** — just changes the API surface from Spark DataFrame to pyspark.pandas; still lazy
3. `psdf.groupby('region').agg({'revenue': 'sum'})` — lazy; adds groupBy+agg to the plan
4. `.to_pandas()` — **triggers execution** on the Spark cluster:
   - Spark reads 50 GB of Parquet files (distributed across Executors)
   - Performs the groupBy aggregation (distributed)
   - Result: likely a small number of rows (one row per distinct `region` value)
   - Collects those few rows to the driver
   - Returns a native pandas DataFrame

The key insight: although 50 GB is read, `to_pandas()` only collects the **aggregated result** (e.g., 10 rows, one per region) to the driver — not 50 GB. This is safe.

**Warning case:** If the aggregation was removed: `psdf.to_pandas()` would attempt to collect all 50 GB to the driver → OutOfMemoryError.

**Why the others are wrong:**
- **A** is wrong — `.pandas_api()` causes no data movement; it is a zero-cost API change
- **B** is wrong — groupBy on pyspark.pandas executes on the Spark cluster, not the driver
- **D** is wrong — `pyspark.pandas` uses the **existing active SparkSession** automatically
</details>

---

In [ ]:
# Score Tracker — run this cell to record your self-assessment
# After reading each answer, mark whether you answered correctly

answers = {
    'Q1':  {'correct': 'B', 'topic': 'Architecture — Driver role'},
    'Q2':  {'correct': 'C', 'topic': 'Architecture — Lazy evaluation'},
    'Q3':  {'correct': 'C', 'topic': 'Architecture — Shuffle partitions'},
    'Q4':  {'correct': 'C', 'topic': 'Architecture — Fault tolerance'},
    'Q5':  {'correct': 'B', 'topic': 'Spark SQL — dense_rank vs rank'},
    'Q6':  {'correct': 'B', 'topic': 'Spark SQL — Predicate pushdown'},
    'Q7':  {'correct': 'B', 'topic': 'Spark SQL — Date/time functions'},
    'Q8':  {'correct': 'B', 'topic': 'Spark SQL — NULL vs NaN'},
    'Q9':  {'correct': 'A', 'topic': 'DataFrame — when/otherwise with NULL'},
    'Q10': {'correct': 'B', 'topic': 'DataFrame — Join row multiplication'},
    'Q11': {'correct': 'B', 'topic': 'DataFrame — coalesce vs repartition'},
    'Q12': {'correct': 'B', 'topic': 'DataFrame — UDF NULL handling'},
    'Q13': {'correct': 'C', 'topic': 'DataFrame — Dynamic partition overwrite'},
    'Q14': {'correct': 'C', 'topic': 'DataFrame — agg() return type'},
    'Q15': {'correct': 'B', 'topic': 'Troubleshooting — Data skew'},
    'Q16': {'correct': 'C', 'topic': 'Performance — Broadcast join'},
    'Q17': {'correct': 'B', 'topic': 'Streaming — Output modes'},
    'Q18': {'correct': 'C', 'topic': 'Streaming — Watermark late data'},
    'Q19': {'correct': 'B', 'topic': 'Spark Connect — Connection syntax'},
    'Q20': {'correct': 'C', 'topic': 'Pandas API — to_pandas() behavior'},
}

# Replace each None with 'A', 'B', 'C', or 'D' for your self-assessment
my_answers = {
    'Q1':  None,
    'Q2':  None,
    'Q3':  None,
    'Q4':  None,
    'Q5':  None,
    'Q6':  None,
    'Q7':  None,
    'Q8':  None,
    'Q9':  None,
    'Q10': None,
    'Q11': None,
    'Q12': None,
    'Q13': None,
    'Q14': None,
    'Q15': None,
    'Q16': None,
    'Q17': None,
    'Q18': None,
    'Q19': None,
    'Q20': None,
}

# --- Score calculation ---
answered = {q: a for q, a in my_answers.items() if a is not None}
if not answered:
    print('Fill in your answers in my_answers{} above, then re-run this cell.')
else:
    correct = sum(1 for q, a in answered.items() if a == answers[q]['correct'])
    total   = len(answered)
    pct     = correct / total * 100

    print(f'Score: {correct}/{total} ({pct:.1f}%)')
    print()
    wrong = [(q, a, answers[q]['correct'], answers[q]['topic'])
             for q, a in answered.items() if a != answers[q]['correct']]
    if wrong:
        print('Review these topics:')
        for q, given, correct_ans, topic in wrong:
            print(f'  {q}: you answered {given}, correct is {correct_ans} — {topic}')
    else:
        print('Perfect score! Great prep!')

## Common Developer Mistakes

<details>
<summary>Mistake 1: Calling collect() or toPandas() on a large DataFrame</summary>

**Pattern:**
```python
# Developer wants to inspect data
all_data = df.collect()         # ← collects ALL rows to driver
# OR
pdf = df.toPandas()             # ← same problem
# OR
pdf = psdf.to_pandas()          # ← pyspark.pandas version
```

**Problem:** If `df` has millions of rows, `collect()` transfers all data to the driver's JVM heap, causing OutOfMemoryError.

**Fix:**
```python
df.show(20)             # displays 20 rows in tabular format — does NOT collect all
df.limit(100).show()    # inspect up to 100 rows
sample = df.sample(fraction=0.01).toPandas()  # 1% random sample — safe

# For aggregations — collect only the small result:
result = df.groupBy('category').count().toPandas()  # only N_categories rows collected
```

**Rule:** Only call `collect()` or `toPandas()` when you are certain the result is small (e.g., after aggregation, after `.limit()`, after `.sample()`).
</details>

<details>
<summary>Mistake 2: Using Python string formatting to build SQL queries with user input</summary>

**Pattern (SQL injection vulnerability):**
```python
# Developer receives user input and directly interpolates it
user_region = request.args.get('region')  # e.g., user inputs: "USA' OR '1'='1"

query = f"SELECT * FROM sales WHERE region = '{user_region}'"
spark.sql(query)  # ← SQL injection risk!
```

**Problem:** User can inject arbitrary SQL.

**Fix — use parameterised queries or the DataFrame API:**
```python
# Option 1: DataFrame API (no SQL injection possible)
from pyspark.sql.functions import col, lit

user_region = request.args.get('region')
df.filter(col('region') == lit(user_region))  # safe — literal comparison

# Option 2: Validate/whitelist the input
ALLOWED_REGIONS = {'USA', 'UK', 'CA', 'AU'}
if user_region not in ALLOWED_REGIONS:
    raise ValueError(f'Invalid region: {user_region}')
df.filter(col('region') == user_region)
```
</details>

<details>
<summary>Mistake 3: Not caching a DataFrame that is used in multiple actions</summary>

**Pattern:**
```python
df_clean = spark.read.parquet('/data/') \
    .filter(col('valid') == True) \
    .withColumn('revenue', col('price') * col('qty'))

count   = df_clean.count()       # reads /data/ and recomputes lineage
summary = df_clean.describe()    # reads /data/ AGAIN and recomputes
df_clean.write.parquet('/out/')  # reads /data/ a THIRD time and recomputes
```

**Problem:** Without caching, every action re-executes the full lineage. The data is read from disk and filtered three times.

**Fix:**
```python
df_clean = spark.read.parquet('/data/') \
    .filter(col('valid') == True) \
    .withColumn('revenue', col('price') * col('qty'))

df_clean.cache()  # ← or .persist(StorageLevel.MEMORY_AND_DISK)

count   = df_clean.count()       # first action: reads /data/ and populates cache
summary = df_clean.describe()    # reads from cache — fast
df_clean.write.parquet('/out/')  # reads from cache — fast

df_clean.unpersist()  # always unpersist when done to free memory
```
</details>

<details>
<summary>Mistake 4: Forgetting that sort_values() in pyspark.pandas triggers a full global shuffle</summary>

**Pattern:**
```python
import pyspark.pandas as ps

psdf = ps.read_parquet('/data/large_dataset/')

# Developer wants to see the 5 most expensive items
top5 = psdf.sort_values('price', ascending=False).head(5)
# This works but sorts ALL data globally (full shuffle) just to get 5 rows
```

**Problem:** `sort_values()` on a distributed DataFrame triggers a full global sort (expensive shuffle). To get the top-N rows, this is extremely wasteful.

**Fix — use nlargest() instead of sort then head:**
```python
# Option 1: nlargest() / nsmallest() — optimised for top-N
top5 = psdf.nlargest(5, 'price')  # much more efficient — no full sort needed
top5_asc = psdf.nsmallest(5, 'price')

# Option 2: Use native PySpark for top-N
from pyspark.sql.functions import desc
top5 = psdf.to_spark() \
    .orderBy(desc('price')) \
    .limit(5)
# PySpark can optimise orderBy+limit better than sort_values+head
```
</details>

<details>
<summary>Mistake 5: Using == to compare NULL values in Spark</summary>

**Pattern:**
```python
# Developer wants to find rows where 'region' is NULL
df.filter(col('region') == None)       # ← returns ZERO rows!
df.filter(col('region') == 'null')     # ← looks for the string 'null', not SQL NULL
df.filter(col('region').equals(None))  # ← not a valid Spark Column method
```

**Problem:** In SQL (and Spark), `NULL == NULL` evaluates to `NULL` (unknown), not `True`. The filter treats NULL comparisons as false, so `col('region') == None` matches zero rows.

**Fix:**
```python
from pyspark.sql.functions import col, isnull, isnan

# Check for NULL:
df.filter(col('region').isNull())          # ← correct
df.filter(isnull(col('region')))           # ← also correct

# Check for NOT NULL:
df.filter(col('region').isNotNull())       # ← correct

# In SQL:
spark.sql("SELECT * FROM t WHERE region IS NULL")      # ← correct
spark.sql("SELECT * FROM t WHERE region IS NOT NULL")  # ← correct
spark.sql("SELECT * FROM t WHERE region = NULL")       # ← WRONG — returns nothing

# NULL-safe equality (matches NULL == NULL as True):
df.filter(col('region').eqNullSafe(None))              # ← True if region is NULL
df.filter(col('region').eqNullSafe('USA'))             # ← True if region = 'USA' OR both NULL
```
</details>